# DSA 504 — Class 12
## SQL Fundamentals

**Date:** Tuesday, Oct 14 (no class Oct 12 — mid-semester break)
**Reading:** *Practical Business Analytics Using R and Python*, ch. 3
**Today:** HW3 is due before class.

---

Everything so far has lived in a file (CSV, JSON, Excel, Parquet) or arrived from an API. Today covers a third way data lives: inside a **relational database**, queried with **SQL** (Structured Query Language) — the standard language for asking questions of data stored in tables.

### Learning goals
By the end of this class, you will be able to:
- Explain what a table, a row, a column, and a primary key are
- Write `SELECT` statements to retrieve specific columns and rows
- Filter rows with `WHERE`, including multiple combined conditions
- Sort results with `ORDER BY`
- Aggregate data with `GROUP BY` and functions like `SUM`, `AVG`, `COUNT`
- Run SQL queries from Python and get the result back as a pandas DataFrame


In [1]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("retail_sales.db")

# A quick first look -- how many tables exist in this database?
tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table'", conn)
print(tables)


     name
0   sales
1  stores


## 1. Tables, Rows, and Keys

A relational database stores data in **tables** — rows and columns, conceptually similar to a pandas DataFrame, but living inside a database file instead of in memory.

This database has two tables:
- `sales` — the same retail sales data we've used all semester (4,014 rows)
- `stores` — a small lookup table with one row per store, giving its region and the year it opened

**A primary key** is a column (or combination of columns) that uniquely identifies each row in a table — in `stores`, that's the `store` column, since no two rows share the same store name. We'll use this concept more directly next class (Class 13), when we **join** `sales` and `stores` together.


In [2]:
stores = pd.read_sql_query("SELECT * FROM stores", conn)
stores


,store,region,opened_year
0,Utica,Central NY,2014
1,Albany,Capital District,2011
2,Rome,Central NY,2018
3,Syracuse,Central NY,2016


## 2. SELECT — Choosing Columns

The most basic SQL query retrieves data with `SELECT`. `SELECT *` means "every column."


In [3]:
query = "SELECT * FROM sales LIMIT 5"
result = pd.read_sql_query(query, conn)
result


,date,store,units_sold,revenue,category
0,2025-04-22,Rome,25.0,402.50,toys
1,2025-04-12,Syracuse,38.0,3154.00,electronics
2,2025-12-21,Utica,42.0,1299.06,home goods
3,2025-02-11,Syracuse,60.0,5114.40,electronics
4,2025-04-10,Syracuse,10.0,147.10,toys


**`LIMIT`** restricts how many rows come back — essential when exploring a table you've never seen before, so you don't accidentally pull millions of rows just to look at a few.

Selecting specific columns, instead of everything:


In [4]:
query = "SELECT store, category, revenue FROM sales LIMIT 5"
result = pd.read_sql_query(query, conn)
result


,store,category,revenue
0,Rome,toys,402.50
1,Syracuse,electronics,3154.00
2,Utica,home goods,1299.06
3,Syracuse,electronics,5114.40
4,Syracuse,toys,147.10


**Connecting this to pandas:** `SELECT store, category, revenue FROM sales` is the SQL equivalent of `sales[["store", "category", "revenue"]]` — same idea, different language.


## 3. WHERE — Filtering Rows

`WHERE` filters which rows come back, the SQL equivalent of boolean indexing in pandas.


In [5]:
query = """
SELECT store, category, revenue
FROM sales
WHERE revenue > 3000
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,store,category,revenue
0,Syracuse,electronics,3154.00
1,Syracuse,electronics,5114.40
2,Utica,electronics,3531.06
3,Syracuse,electronics,3009.60
4,Utica,electronics,3674.55


### Combining conditions with AND / OR

Unlike pandas (where you use `&` and `|`), SQL uses the plain English words `AND` and `OR`.


In [6]:
query = """
SELECT store, category, revenue
FROM sales
WHERE store = 'Utica' AND revenue > 2000
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
print(f"Rows matching: {len(pd.read_sql_query(query.replace('LIMIT 5', ''), conn))}")
result


Rows matching: 216


,store,category,revenue
0,Utica,electronics,3531.06
1,Utica,electronics,3674.55
2,Utica,electronics,6814.08
3,Utica,electronics,3312.27
4,Utica,electronics,2727.36


In [7]:
# OR, and combining with IN (SQL's equivalent of pandas' .isin())
query = """
SELECT store, category, revenue
FROM sales
WHERE store IN ('Utica', 'Rome')
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,store,category,revenue
0,Rome,toys,402.50
1,Utica,home goods,1299.06
2,Utica,electronics,3531.06
3,Rome,groceries,807.54
4,Utica,home goods,921.62


**A note on quotes:** SQL text values go in single quotes (`'Utica'`), not double quotes. This trips people up constantly coming from Python, where both usually work interchangeably.


## 4. ORDER BY — Sorting Results


In [8]:
query = """
SELECT store, category, revenue
FROM sales
ORDER BY revenue DESC
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,store,category,revenue
0,Rome,electronics,7021.63
1,Utica,electronics,6814.08
2,Albany,electronics,6395.40
3,Albany,electronics,6259.36
4,Utica,electronics,6120.10


`DESC` means descending (highest first); leave it off, or write `ASC`, for ascending (the default).


## 5. GROUP BY — Aggregating Data

Just like pandas' `.groupby()`, SQL's `GROUP BY` computes a statistic per group instead of one overall number.


In [9]:
query = """
SELECT store, SUM(revenue) AS total_revenue
FROM sales
GROUP BY store
ORDER BY total_revenue DESC
"""
result = pd.read_sql_query(query, conn)
result


,store,total_revenue
0,Utica,1562319.71
1,Rome,1521763.90
2,Syracuse,1518656.06
3,Albany,1477075.47


**`AS`** renames a column in the result — here, `SUM(revenue)` becomes the more readable `total_revenue`.

Common aggregate functions: `SUM()`, `AVG()` (average), `COUNT()`, `MIN()`, `MAX()`.


In [10]:
query = """
SELECT category, AVG(units_sold) AS avg_units, COUNT(*) AS num_transactions
FROM sales
GROUP BY category
ORDER BY avg_units DESC
"""
result = pd.read_sql_query(query, conn)
result


,category,avg_units,num_transactions
0,groceries,128.710843,830
1,clothing,64.328851,818
2,electronics,43.859008,766
3,home goods,37.650185,809
4,toys,26.305942,791


**Grouping by more than one column:**


In [11]:
query = """
SELECT store, category, SUM(revenue) AS total_revenue
FROM sales
GROUP BY store, category
ORDER BY store, total_revenue DESC
LIMIT 10
"""
result = pd.read_sql_query(query, conn)
result


,store,category,total_revenue
0,Albany,electronics,668308.55
1,Albany,clothing,314275.38
2,Albany,home goods,252879.37
3,Albany,groceries,165273.25
4,Albany,toys,76338.92
5,Rome,electronics,680236.42
6,Rome,clothing,338239.18
7,Rome,home goods,241580.28
8,Rome,groceries,176295.41
9,Rome,toys,85412.61


## 6. Filtering Aggregated Results with HAVING

`WHERE` filters rows *before* grouping. To filter *after* grouping — for example, "only show stores whose total revenue exceeds some amount" — SQL uses a different keyword: `HAVING`.


In [12]:
query = """
SELECT store, SUM(revenue) AS total_revenue
FROM sales
GROUP BY store
HAVING SUM(revenue) > 1500000
ORDER BY total_revenue DESC
"""
result = pd.read_sql_query(query, conn)
result


,store,total_revenue
0,Utica,1562319.71
1,Rome,1521763.90
2,Syracuse,1518656.06


**Why not just use `WHERE` here?** `WHERE revenue > 1500000` would filter individual *rows* before any grouping happens — completely different from "keep only the stores whose *total* exceeds that amount." `HAVING` is specifically for conditions on the aggregated result.


## 7. Combining Everything in One Query

Real queries usually chain several of today's clauses together. SQL has a required clause order: `SELECT` → `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `ORDER BY` → `LIMIT`.


In [13]:
query = """
SELECT category, SUM(revenue) AS total_revenue, COUNT(*) AS num_sales
FROM sales
WHERE store = 'Syracuse'
GROUP BY category
HAVING COUNT(*) > 50
ORDER BY total_revenue DESC
"""
result = pd.read_sql_query(query, conn)
result


,category,total_revenue,num_sales
0,electronics,735558.34,196
1,clothing,305867.60,199
2,home goods,231201.53,194
3,groceries,169436.17,197
4,toys,76592.42,191


In [29]:
conn.close()

---
## Guided Practice

Write SQL queries against the `sales` table (and `stores` where noted), using `pd.read_sql_query()` to run them.


### Exercise 1 — SELECT and WHERE
Write a query that selects `date`, `store`, and `units_sold` for all rows where `units_sold` is greater than 100. Limit to 10 rows.


In [14]:
# Exercise 1 — your code here
query = """
SELECT date, store, units_sold
FROM sales
WHERE units_sold > 100
LIMIT 10
"""
result = pd.read_sql_query(query, conn)
result


,date,store,units_sold
0,2025-03-04,Rome,129.0
1,2025-11-26,Utica,108.0
2,2025-09-04,Rome,135.0
3,2025-07-27,Syracuse,186.0
4,2025-05-28,Utica,151.0
5,2025-09-22,Utica,148.0
6,2025-04-24,Albany,179.0
7,2025-12-25,Albany,126.0
8,2025-08-19,Utica,107.0
9,2025-07-30,Albany,116.0


### Exercise 2 — Combined conditions
Write a query that selects all columns where `category = 'electronics'` AND `revenue > 1000`. How many rows match (run the query without `LIMIT` and check the length)?


In [15]:
# Exercise 2 — your code here
query = """
SELECT *
FROM sales
WHERE category = 'electronics' AND revenue > 1000
"""
# To check how many rows match:
matching_rows = pd.read_sql_query(query, conn)
print(f"Number of matching rows: {len(matching_rows)}")
matching_rows.head()


Number of matching rows: 765


,date,store,units_sold,revenue,category
0,2025-04-12,Syracuse,38.0,3154.00,electronics
1,2025-02-11,Syracuse,60.0,5114.40,electronics
2,2025-05-07,Utica,39.0,3531.06,electronics
3,2025-12-22,Rome,35.0,1128.13,electronics
4,2025-08-22,Syracuse,38.0,3009.60,electronics


### Exercise 3 — ORDER BY
Write a query that selects `store`, `category`, and `units_sold`, sorted by `units_sold` ascending (lowest first). Limit to 5 rows.


In [16]:
# Exercise 3 — your code here
query = """
SELECT store, category, units_sold
FROM sales
ORDER BY units_sold ASC
LIMIT 5
"""
result = pd.read_sql_query(query, conn)
result


,store,category,units_sold
0,Syracuse,home goods,4.0
1,Albany,toys,7.0
2,Albany,toys,8.0
3,Syracuse,toys,9.0
4,Syracuse,home goods,9.0


### Exercise 4 — GROUP BY with HAVING
Write a query that computes average revenue per category, but only shows categories where the average exceeds 1000.


In [17]:
# Exercise 4 — your code here
query = """
SELECT category, AVG(revenue) AS avg_revenue
FROM sales
GROUP BY category
HAVING AVG(revenue) > 1000
"""
result = pd.read_sql_query(query, conn)
result


,category,avg_revenue
0,clothing,1542.746699
1,electronics,3689.503969
2,home goods,1199.540111


### Exercise 5 (stretch) — Using the stores table
Write a query against the `stores` table that selects only stores in the `'Central NY'` region, ordered by `opened_year` (earliest first).


In [18]:
# Exercise 5 — your code here
query = """
SELECT *
FROM stores
WHERE region = 'Central NY'
ORDER BY opened_year ASC
"""
result = pd.read_sql_query(query, conn)
result


,store,region,opened_year
0,Utica,Central NY,2014
1,Syracuse,Central NY,2016
2,Rome,Central NY,2018


---
## Solutions

Try each exercise yourself first. These are here for after class, or once you're stuck.


---
## Wrap-up

**Recap:** tables, rows, and primary keys; `SELECT` and `LIMIT`; filtering with `WHERE` (including `AND`/`OR`/`IN`); sorting with `ORDER BY`; aggregating with `GROUP BY` and `SUM`/`AVG`/`COUNT`; filtering aggregated results with `HAVING`; and running SQL queries from Python with `pd.read_sql_query()`, which hands results back as a familiar DataFrame.

**Common mistakes to watch for:**
- Using double quotes instead of single quotes around text values (`"Utica"` vs `'Utica'`)
- Using `WHERE` when you meant `HAVING` (or vice versa) — remember, `WHERE` filters rows before grouping, `HAVING` filters groups after
- Forgetting `GROUP BY` when using an aggregate function alongside a non-aggregated column — SQL will often error, unlike pandas, which is more forgiving about this
- Forgetting to close the database connection (`conn.close()`) once you're finished

**HW3 is due before this class.**

**Before next class (Oct 19):** Class 13 covers SQL joins — combining data from `sales` and `stores` together, using the `store` column as the shared key between them.
